In [6]:
import matplotlib.pyplot as plt
import csv
from PIL import Image
import numpy as np
import os
import pickle
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from skimage.color import rgb2gray
from skimage import exposure
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


## Defining Function

read_Image_and_CSV_Data() is used to read images and 4 outputs and pre-process them. It will return a list of images (NumPy arrays), NumPy array of corresponding outputs (N, 4)

In [ ]:
# Column indices in the CSV file for the 4 outputs 
X_COL, Y_COL, XW_COL, YW_COL = 1, 2, 3, 4 


def read_Image_and_CSV_Data(rootpath):
 
    images = []
    all_outputs = []
    prefix = rootpath + '/' 
    
    try:
        with open(prefix + 'myData'+ '.csv', 'r') as gtFile:
            gtReader = csv.reader(gtFile, delimiter=';')
            next(gtReader) # Skip header row
            

            for row in gtReader:
                if not row or len(row) <= YW_COL:
                    continue
                
                # Image Loading
                img_path = prefix + row[0]
                img = Image.open(img_path)

                # Resize to 32x32
                img = img.resize((32, 32), Image.BICUBIC) 
                img_array = np.array(img)

                # Convert to grayscale
                gray_img = rgb2gray(img_array)
                
                # Apply CLAHE (Contrast Enhancement)
                clache_img = exposure.equalize_adapthist(gray_img, clip_limit=0.03)
                
                images.append(clache_img)
                
                # Read all 4 outputs
                x = int(row[X_COL])
                y = int(row[Y_COL])
                xw = int(row[XW_COL])
                yw = int(row[YW_COL])
                all_outputs.append([x, y, xw, yw])

    except FileNotFoundError:
        print(f"Error: Training file not found at {prefix + 'myData.csv'}")
        exit()
    except Exception as e:
        print(f"Skipping row due to error: {e}")
        
    outputs_array = np.array(all_outputs)
    return images, outputs_array

## Main Execution Code

1. Load and Prepare Train Data

In [ ]:
#load the raw data
trainImages, trainOutputs = read_Image_and_CSV_Data('Training')
print('number of historical data=', len(trainOutputs))


# Create model directory for saving models
os.makedirs('saved_models', exist_ok=True)

# design the input and output for model
X = []
Y = trainOutputs.copy()

for i in range(0,len(trainOutputs)):
    
    # input X just the flattern image
    X.append(trainImages[i].flatten())

X = np.array(X)

print('Shape of feature matrix X:', X.shape) 

number of historical data= 150
Shape of feature matrix X: (150, 1024)


2. Train Random Forest

In [9]:

# --- Model: Random Forest Regressor ---
print("\n--- Training Random Forest Regressor ---")

# RFR natively supports multi-output regression
rf_reg = RandomForestRegressor(n_estimators=100, max_features=None, random_state=42) 
rf_reg.fit(X, Y)

# Predict on training data
Y_pred_rf = rf_reg.predict(X)

# Check the accuracy (Training MSE)
mse_rf = mean_squared_error(Y, Y_pred_rf)
print('Random Forest Training MSE (on 4 outputs):', mse_rf)

# Save the Random Forest model
pickle.dump(rf_reg, open('saved_models/rf_model.sav', 'wb'))
print("Saved Random Forest model to 'saved_models/rf_model.sav'")

# Save Random Forest Prediction Results
pred_filename_rf = 'saved_models/rf_predictions_train.csv'
np.savetxt(pred_filename_rf, Y_pred_rf, delimiter=',', header='x_pred,y_pred,xw_pred,yw_pred', comments='')
print(f"Saved Random Forest predictions to '{pred_filename_rf}'")


--- Training Random Forest Regressor ---
Random Forest Training MSE (on 4 outputs): 262.2823201666665
Saved Random Forest model to 'saved_models/rf_model.sav'
Saved Random Forest predictions to 'saved_models/rf_predictions_train.csv'


3. Train SVR

In [10]:
print("\n--- Training Support Vector Regressor (SVR) ---")

# Define the kernel and C value of our SVR model
base_svr = SVR(kernel='linear', C=2.1544)

# wrap 4 independent SVR models using MultiOutputRegressor
multioutput_svr = MultiOutputRegressor(base_svr)

# Train the model
multioutput_svr.fit(X, Y)

# Predict on training data
Y_pred_svr = multioutput_svr.predict(X)

# Check the accuracy (Training MSE)
mse_svr = mean_squared_error(Y, Y_pred_svr)
print('SVR Training MSE (on 4 outputs):', mse_svr)

# Save the SVR model
pickle.dump(multioutput_svr, open('saved_models/svr_model.sav', 'wb'))
print("Saved SVR model to 'saved_models/svr_model.sav'")

# Save SVR Prediction Results
pred_filename_svr = 'saved_models/svr_predictions_train.csv'
np.savetxt(pred_filename_svr, Y_pred_svr, delimiter=',', header='x_pred,y_pred,xw_pred,yw_pred', comments='')
print(f"Saved SVR predictions to '{pred_filename_svr}'")


--- Training Support Vector Regressor (SVR) ---
SVR Training MSE (on 4 outputs): 64.14594803126245
Saved SVR model to 'saved_models/svr_model.sav'
Saved SVR predictions to 'saved_models/svr_predictions_train.csv'
